In [ ]:
!pip install transformers datasets scikit-learn torch pandas numpy

In [ ]:
from datasets import load_dataset

dataset = load_dataset("liamdugan/raid", split="train")
print(dataset)

In [ ]:
import pandas as pd

# Load more rows to get enough human samples
df = dataset.select(range(400000)).to_pandas()

del dataset
import gc
gc.collect()

print(df.shape)
print(df['model'].value_counts())

In [ ]:
# Save balanced dataset
df_balanced.to_csv('raid_balanced.csv', index=False)

# Download to your laptop
from google.colab import files
files.download('raid_balanced.csv')
print("Downloaded! 19000 AI + 19000 Human = 38000 samples")

In [ ]:
from sklearn.model_selection import train_test_split
import gc

df = df[['generation', 'model', 'domain']].dropna()
df = df.rename(columns={'generation': 'text'})
df['label'] = df['model'].apply(lambda x: 0 if x == 'human' else 1)

# Filter only abstracts domain for better real world testing
df = df[df['domain'] == 'abstracts']

human_count = len(df[df['label'] == 0])
ai_count = len(df[df['label'] == 1])
print(f"Human: {human_count}, AI: {ai_count}")

sample_size = min(human_count, ai_count, 19000)
print(f"Sampling: {sample_size} each")

ai = df[df['label'] == 1].sample(sample_size, random_state=42)
human = df[df['label'] == 0].sample(sample_size, random_state=42)
df_balanced = pd.concat([ai, human]).sample(frac=1, random_state=42).reset_index(drop=True)

del df
gc.collect()

train_df, test_df = train_test_split(df_balanced, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)

print(f"Total balanced: {len(df_balanced)}")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained('bert-base-uncased')

def tokenize(texts, max_len=128):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

train_encodings = tokenize(train_df['text'])
val_encodings   = tokenize(val_df['text'])
test_encodings  = tokenize(test_df['text'])

print("Tokenization complete!")

In [ ]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = TextDataset(train_encodings, list(train_df['label']))
val_dataset   = TextDataset(val_encodings,   list(val_df['label']))
test_dataset  = TextDataset(test_encodings,  list(test_df['label']))
print("Dataset created!")

In [ ]:
from transformers import RobertaForSequenceClassification

model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=2
)
print("RoBERTa Model loaded!")

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("Accuracy:", accuracy_score(labels, preds))
print(classification_report(labels, preds, target_names=['Human', 'AI']))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained('/content/drive/MyDrive/roberta_mgd_model')
tokenizer.save_pretrained('/content/drive/MyDrive/roberta_mgd_model')
print("Model saved!")

In [ ]:
def predict_text(text):
    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    ).to('cuda' if torch.cuda.is_available() else 'cpu')

    model.to('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=1).item()
        confidence = torch.softmax(outputs.logits, dim=1).max().item()

    label = "🤖 AI Generated" if prediction == 1 else "👤 Human Written"
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence*100:.2f}%")

# Interactive input
while True:
    text = input("\nEnter text to check (or type 'quit' to stop): ")
    if text.lower() == 'quit':
        print("Exiting...")
        break
    predict_text(text)

In [ ]:
print(train_df['label'].value_counts())
print(train_df['label'].dtype)
print(train_df.head(3))

In [ ]:
print(type(tokenizer))